# PREREQUISITE

## Download dependency

In [1]:
ls -a

./  ../  .config/  sample_data/


In [2]:

!pip install pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.5 MB/s eta 0:00:00


## Library import

In [3]:
import os
import numpy as np
import pandas as pd
import random
from transformers import AutoTokenizer, get_linear_schedule_with_warmup, AutoModel, get_cosine_schedule_with_warmup
import re
import unicodedata
from collections import Counter
from google.colab import files
from torch.optim.lr_scheduler import StepLR
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

## Config value

In [4]:
"""
Central config for the Vietnamese ABSA project (UIT-ViSFD -> transfer to MoMo).
"""

# ---- Paths ----
# PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
# DATA_DIR = os.path.join(PROJECT_ROOT, "data")
# CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

# TRAIN_CSV = os.path.join(DATA_DIR, "Train.csv")
# DEV_CSV = os.path.join(DATA_DIR, "Dev.csv")
# TEST_CSV = os.path.join(DATA_DIR, "Test.csv")
DATA_URL = "https://raw.githubusercontent.com/NVKQ2022/VN-ABSA/main/data"
TRAIN_CSV = f"{DATA_URL}/Train.csv"
DEV_CSV = f"{DATA_URL}/Dev.csv"
TEST_CSV =f"{DATA_URL}/Test.csv"
#CHECKPOINT_DIR = "checkpoints"

# ---- Label space (UIT-ViSFD) ----
# 10 aspect categories, each with 4 possible states: None / Positive / Negative / Neutral
ASPECTS = [
    "BATTERY", "CAMERA", "DESIGN", "FEATURES", "GENERAL",
    "PERFORMANCE", "PRICE", "SCREEN", "SER&ACC", "STORAGE",
]

POLARITY2ID = {"None": 0, "Positive": 1, "Negative": 2, "Neutral": 3}
ID2POLARITY = {v: k for k, v in POLARITY2ID.items()}
NUM_POLARITY_CLASSES = len(POLARITY2ID)  # 4
NUM_ASPECTS = len(ASPECTS)  # 10

# ---- Model ----
MODEL_NAME = "vinai/phobert-base-v2"
MAX_LEN = 256
DROPOUT = 0.2

# ---- Training ----
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 4e-6 #last best 2e-6 # start high but later will lower by lr_scheduler (2e-5)
NUM_EPOCHS = 5
WARMUP_RATIO = 0.15
WEIGHT_DECAY = 0.01
SEED = 42
GRAD_CLIP_NORM = 1.0

# early stopping on dev macro-F1 (averaged across aspects, sentiment-level)
EARLY_STOPPING_PATIENCE = 3

DEVICE = "cuda"  # falls back to cpu automatically in train.py if unavailable

# ---- Continue training ----
CONTINUE_TRAINING = True

## Preprocessing

In [5]:
"""
Preprocessing utilities:
  - parse_label_string: turns "{CAMERA#Positive};{BATTERY#Negative};{OTHERS};"
    into a dict {aspect: polarity} over the fixed ASPECTS list.
  - clean_text / segment_text: text normalization + PhoBERT word segmentation.
"""
import re
import unicodedata

# from config import ASPECTS, POLARITY2ID

_TAG_RE = re.compile(r"\{([^}]*)\}")

_word_segmenter = None  # lazy-loaded pyvi tokenizer (avoids import cost if unused)


def parse_label_string(label_str: str) -> dict:
    """
    Parse a raw UIT-ViSFD label string into {aspect: polarity_id}.
    Aspects not mentioned default to POLARITY2ID["None"].
    The bare {OTHERS} tag carries no polarity and is ignored here;
    see `has_others_tag` if you want to use it as an extra signal.
    """
    result = {aspect: POLARITY2ID["None"] for aspect in ASPECTS}
    if not isinstance(label_str, str):
        return result
    for tag in _TAG_RE.findall(label_str):
        if "#" not in tag:
            continue  # e.g. bare "OTHERS"
        aspect, polarity = tag.split("#", 1)
        aspect = aspect.strip()
        polarity = polarity.strip()
        if aspect in result and polarity in POLARITY2ID:
            result[aspect] = POLARITY2ID[polarity]
    return result


def has_others_tag(label_str: str) -> bool:
    if not isinstance(label_str, str):
        return False
    return any(tag.strip() == "OTHERS" for tag in _TAG_RE.findall(label_str))


def clean_text(text: str) -> str:
    """Light normalization: unicode NFC, collapse whitespace, strip."""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def segment_text(text: str) -> str:
    """
    Vietnamese word segmentation required for PhoBERT
    (e.g. "đăng nhập" -> "đăng_nhập"). Uses pyvi; loaded lazily.
    """
    global _word_segmenter
    if _word_segmenter is None:
        from pyvi import ViTokenizer
        _word_segmenter = ViTokenizer
    return _word_segmenter.tokenize(text)


def preprocess_for_model(text: str) -> str:
    """Full pipeline applied before tokenization: clean -> segment."""
    return segment_text(clean_text(text))


## Dataset class

In [6]:
"""
PyTorch Dataset for the multi-aspect multi-sentiment classification task.

Each item returns:
  input_ids, attention_mask  (from PhoBERT tokenizer)
  labels: LongTensor of shape (NUM_ASPECTS,) with values in POLARITY2ID
"""
from collections import Counter

import pandas as pd
import torch
from torch.utils.data import Dataset

# from config import ASPECTS, ID2POLARITY, MAX_LEN, POLARITY2ID
# from preprocessing import parse_label_string, preprocess_for_model, clean_text

class ABSADataset(Dataset):
    # def __init__(self, csv_path, tokenizer, max_len=MAX_LEN, segment=True):
    #     df = pd.read_csv(csv_path)
    #     assert {"comment", "label"}.issubset(df.columns), \
    #         f"{csv_path} missing required columns"
    #     df = df.dropna(subset=["comment", "label"]).reset_index(drop=True)

    #     self.tokenizer = tokenizer
    #     self.max_len = max_len
    #     self.segment = segment

    #     self.texts = df["comment"].apply(clean_text).tolist()
    #     self.label_dicts = df["label"].apply(parse_label_string).tolist()
    """
    Tokenize + word-segment MỘT LẦN DUY NHẤT trong __init__, cache sẵn
    input_ids/attention_mask/labels dưới dạng tensor. __getitem__ chỉ còn
    việc index vào tensor có sẵn -> gần như miễn phí về CPU.

    Đánh đổi: tốn thêm RAM để giữ tensor đã tokenize trong bộ nhớ suốt
    quá trình train. Với MAX_LEN=256 và ~7,786 mẫu train, chi phí này chỉ
    khoảng vài chục MB — rất đáng đánh đổi lấy tốc độ.
    """

    def __init__(self, csv_path, tokenizer, max_len=MAX_LEN, segment=True,
                 show_progress=True):
        df = pd.read_csv(csv_path)
        assert {"comment", "label"}.issubset(df.columns), \
            f"{csv_path} thiếu cột comment/label"
        df = df.dropna(subset=["comment", "label"]).reset_index(drop=True)

        self.max_len = max_len
        raw_texts = df["comment"].apply(clean_text).tolist()
        label_dicts = df["label"].apply(parse_label_string).tolist()

        # ---- Bước tốn thời gian nhất: word segmentation, chạy 1 lần duy nhất ----
        if segment:
            iterator = raw_texts
            if show_progress:
                try:
                    from tqdm.auto import tqdm
                    iterator = tqdm(raw_texts, desc="Word-segmenting (pyvi)")
                except ImportError:
                    pass
            processed_texts = [segment_text(t) for t in iterator]
        else:
            processed_texts = raw_texts

        # ---- Tokenize toàn bộ 1 lần bằng batch encode ----
        # (nhanh hơn nhiều so với gọi tokenizer() từng câu trong __getitem__)
        encodings = tokenizer(
            processed_texts,
            truncation=True,
            max_length=max_len,
            padding="max_length",
            return_tensors="pt",
        )
        self.input_ids = encodings["input_ids"]            # (N, max_len)
        self.attention_mask = encodings["attention_mask"]  # (N, max_len)

        self.labels = torch.tensor(
            [[d[a] for a in ASPECTS] for d in label_dicts], dtype=torch.long
        )  # (N, num_aspects) — cache luôn, khỏi build lại mỗi lần



    # def __len__(self):
    #     return len(self.texts)

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, idx):
        # text = self.texts[idx]
        # if self.segment:
        #     text = preprocess_for_model(text)  # clean_text already applied above

        # encoding = self.tokenizer(
        #     text,
        #     truncation=True,
        #     max_length=self.max_len,
        #     padding="max_length",
        #     return_tensors="pt",
        # )

        # label_dict = self.label_dicts[idx]
        # labels = torch.tensor([label_dict[a] for a in ASPECTS], dtype=torch.long)

        # return {
        #     "input_ids": encoding["input_ids"].squeeze(0),
        #     "attention_mask": encoding["attention_mask"].squeeze(0),
        #     "labels": labels,
        # }
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

    def get_info(self):
        """Return general dataset info: aspects, sentiments, counts and ratios."""
        total = len(self)
        per_aspect = {}
        for aspect in ASPECTS:
            counts = Counter(d[aspect] for d in self.label_dicts)
            dist = {ID2POLARITY[pid]: n for pid, n in sorted(counts.items())}
            per_aspect[aspect] = {
                "counts": dist,
                "percent": {pol: round(n / total * 100, 2) for pol, n in dist.items()},
            }
        mention_counts = [
            sum(1 for a in d.values() if a != POLARITY2ID["None"])
            for d in self.label_dicts
        ]
        return {
            "total_samples": total,
            "aspects": list(ASPECTS),
            "sentiments": list(ID2POLARITY.values()),
            "per_aspect": per_aspect,
            "avg_aspects_per_comment": round(sum(mention_counts) / total, 2),
            "comments_with_no_aspect": sum(1 for c in mention_counts if c == 0),
            "max_len": self.max_len,
        }


def compute_class_weights(csv_path):
    """
    Per-aspect class weights (inverse frequency) to counter the heavy
    'None' imbalance on every head. Returns a dict: aspect -> FloatTensor(4,)
    """
    import numpy as np
    df = pd.read_csv(csv_path).dropna(subset=["label"])
    label_dicts = df["label"].apply(parse_label_string).tolist()

    weights = {}
    for aspect in ASPECTS:
        counts = np.zeros(4)
        for d in label_dicts:
            counts[d[aspect]] += 1
        counts = np.clip(counts, 1, None)  # avoid div-by-zero for unseen classes
        inv = 1.0 / counts
        norm = inv / inv.sum() * 4  # normalize so weights average to ~1
        weights[aspect] = torch.tensor(norm, dtype=torch.float)
    return weights


## Model class

In [7]:
import torch
import torch.nn as nn
from transformers import AutoModel

class PhoBertABSA(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=DROPOUT):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        # one independent head per aspect
        self.heads = nn.ModuleDict({
            aspect: nn.Linear(hidden_size, NUM_POLARITY_CLASSES)
            for aspect in ASPECTS
        })

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # use pooled [CLS] representation
        if getattr(outputs, "pooler_output", None) is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0, :]  # fallback: CLS token
        pooled = self.dropout(pooled)

        # (batch, num_aspects, num_polarity_classes)
        logits = torch.stack(
            [self.heads[aspect](pooled) for aspect in ASPECTS], dim=1
        )
        return logits

## Loss function

In [8]:
def compute_loss(logits, labels, class_weights=None):
    """
    logits: (batch, num_aspects, 4)
    labels: (batch, num_aspects)
    class_weights: optional dict aspect -> FloatTensor(4,) on the same device
    """
    total_loss = 0.0
    for i, aspect in enumerate(ASPECTS):
        weight = class_weights[aspect] if class_weights is not None else None
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        total_loss = total_loss + loss_fn(logits[:, i, :], labels[:, i])
    return total_loss / len(ASPECTS)


## Train function

In [9]:
# # def train_model(epochs , model, train_loader, optimizer, scheduler, device, class_weights):
# #     model.train()
# #     for epoch in range(epochs):
# #         total_loss = 0.0
# #         for batch_idx, batch in enumerate(train_loader):
# #             input_ids = batch["input_ids"].to(device)
# #             attention_mask = batch["attention_mask"].to(device)
# #             labels = batch["labels"].to(device)

# #             optimizer.zero_grad()

# #             logits = model(input_ids, attention_mask)
# #             loss = compute_loss(logits, labels, class_weights)

# #             loss.backward()
# #             torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
# #             optimizer.step()
# #             scheduler.step()

# #             total_loss += loss.item()
# #             print(f"Epoch {epoch + 1}/{epochs} | Batch {batch_idx + 1}/{len(train_loader)} | Batch Loss: {loss.item():.4f}")

# #         avg_loss = total_loss / len(train_loader)
# #         print(f"Epoch {epoch + 1}/{epochs}, Average Training Loss: {avg_loss:.4f}")

# import time
# import torch

# def train_model(
#     model, train_loader, dev_loader, optimizer, scheduler, device,
#     class_weights, num_epochs, checkpoint_path="best_model.pt",
#     early_stopping_patience=3, grad_accum_steps=1, log_every=50,
#     use_amp=True,
# ):
#     """
#     Vòng lặp train tối ưu tốc độ, giữ nguyên độ chính xác:

#     - Mixed precision (fp16 autocast + GradScaler): train nhanh hơn ~1.5-2x
#       trên GPU hỗ trợ fp16 (T4/V100/A100...), giảm VRAM dùng. GradScaler tự
#       động scale gradient để tránh underflow -> không mất độ chính xác so
#       với train full fp32.
#     - Gradient accumulation: cho phép effective batch size lớn hơn mà không
#       tốn thêm VRAM (vd batch_size=16, grad_accum_steps=2 -> effective 32).
#     - non_blocking=True khi chuyển tensor lên GPU (cần pin_memory=True ở
#       DataLoader) -> giấu độ trễ copy CPU->GPU phía sau tính toán.
#     - Đánh giá trên dev set sau MỖI epoch (thay vì không đánh giá gì), lưu
#       checkpoint tốt nhất theo macro-F1, và early-stop nếu không cải thiện
#       sau `early_stopping_patience` epoch liên tiếp -> tránh train dư,
#       vừa tốn thời gian vừa tăng nguy cơ overfit.
#     - torch.backends.cudnn.benchmark=True: cudnn tự chọn thuật toán conv
#       nhanh nhất cho input shape cố định (ít ảnh hưởng transformer nhưng
#       không có hại).
#     """
#     amp_enabled = use_amp and torch.cuda.is_available()
#     scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

#     if torch.cuda.is_available():
#         torch.backends.cudnn.benchmark = True

#     best_f1 = -1.0
#     patience_counter = 0
#     history = []

#     for epoch in range(1, num_epochs + 1):
#         model.train()
#         running_loss = 0.0
#         t0 = time.time()
#         optimizer.zero_grad()

#         for step, batch in enumerate(train_loader):
#             input_ids = batch["input_ids"].to(device, non_blocking=True)
#             attention_mask = batch["attention_mask"].to(device, non_blocking=True)
#             labels = batch["labels"].to(device, non_blocking=True)

#             with torch.autocast(device_type="cuda" if amp_enabled else "cpu",
#                                  enabled=amp_enabled, dtype=torch.float16):
#                 logits = model(input_ids, attention_mask)
#                 loss = compute_loss(logits, labels, class_weights) / grad_accum_steps

#             scaler.scale(loss).backward()

#             if (step + 1) % grad_accum_steps == 0:
#                 scaler.unscale_(optimizer)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
#                 scaler.step(optimizer)
#                 scaler.update()
#                 scheduler.step()
#                 optimizer.zero_grad()

#             running_loss += loss.item() * grad_accum_steps
#             if step % log_every == 0:
#                 print(f"epoch {epoch} step {step}/{len(train_loader)} "
#                       f"loss {loss.item() * grad_accum_steps:.4f}")

#         avg_loss = running_loss / len(train_loader)
#         epoch_time = time.time() - t0

#         #dev_f1 = quick_eval(model, dev_loader, device)


#         eval_result = evaluate_detailed(model, dev_loader, device, verbose=False)
#         dev_f1 = eval_result["macro_sentiment_f1_4cls"]
#         history.append({"epoch": epoch, "train_loss": avg_loss,
#                          "dev_macro_f1": dev_f1, "epoch_time_sec": epoch_time})
#         print(f"\n== Epoch {epoch}/{num_epochs} | loss {avg_loss:.4f} "
#               f"| dev macro-F1 {dev_f1:.4f} | {epoch_time:.1f}s ==")

#         if dev_f1 > best_f1:
#             best_f1 = dev_f1
#             patience_counter = 0
#             torch.save(model.state_dict(), checkpoint_path)
#             print(f"  -> lưu best model tại {checkpoint_path} (F1={best_f1:.4f})")
#         else:
#             patience_counter += 1
#             print(f"  -> không cải thiện ({patience_counter}/{early_stopping_patience})")
#             if patience_counter >= early_stopping_patience:
#                 print("Early stopping.")
#                 break

#     print(f"\nHoàn tất. Best dev macro-F1: {best_f1:.4f}")
#     return history

In [10]:
import os
import time
import torch

# def save_checkpoint(path, model, optimizer, scheduler, epoch, best_f1, history, hyperparams):
#     torch.save({
#         "epoch": epoch, "best_f1": best_f1, "history": history,
#         "hyperparams": hyperparams,
#         "model_state": model.state_dict(),
#         "optimizer_state": optimizer.state_dict(),
#         "scheduler_state": scheduler.state_dict(),
#     }, path)
# def save_checkpoint(path, model, optimizer, scheduler, epoch, best_f1, history, hyperparams):
#     os.makedirs(os.path.dirname(path), exist_ok=True)   # <-- thêm dòng này
#     torch.save({
#         "epoch": epoch, "best_f1": best_f1, "history": history,
#         "hyperparams": hyperparams,
#         "model_state": model.state_dict(),
#         "optimizer_state": optimizer.state_dict(),
#         "scheduler_state": scheduler.state_dict(),
#     }, path)

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_f1,
                     history, hyperparams, include_optimizer=True):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ckpt = {
        "epoch": epoch, "best_f1": best_f1, "history": history,
        "hyperparams": hyperparams,
        "model_state": model.state_dict(),
    }
    if include_optimizer:
        ckpt["optimizer_state"] = optimizer.state_dict()
        ckpt["scheduler_state"] = scheduler.state_dict()
    torch.save(ckpt, path)
def load_checkpoint(path, model, optimizer=None, scheduler=None, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    if optimizer is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and "scheduler_state" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    return ckpt.get("epoch", 0), ckpt.get("best_f1", -1.0), ckpt.get("history", [])


def train_model(
    model, train_loader, dev_loader, optimizer, scheduler, device,
    class_weights, num_epochs, checkpoint_path="best_model.pt",
    early_stopping_patience=3, grad_accum_steps=1, log_every=50,
    use_amp=True, resume_from=None, hyperparams=None,
):
    amp_enabled = use_amp and torch.cuda.is_available()
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True

    start_epoch = 1
    best_f1 = -1.0
    history = []

    if resume_from is not None and os.path.exists(resume_from):
        last_epoch, best_f1, history = load_checkpoint(
            resume_from, model, optimizer, scheduler, device
        )
        start_epoch = last_epoch + 1
        print(f"Resume từ checkpoint: epoch {start_epoch}, best_f1 hiện tại = {best_f1:.4f}")

    patience_counter = 0

    for epoch in range(start_epoch, num_epochs + 1):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with torch.autocast(device_type="cuda" if amp_enabled else "cpu",
                                 enabled=amp_enabled, dtype=torch.float16):
                logits = model(input_ids, attention_mask)
                loss = compute_loss(logits, labels, class_weights) / grad_accum_steps

            scaler.scale(loss).backward()
            if (step + 1) % grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            running_loss += loss.item() * grad_accum_steps
            if step % log_every == 0:
                print(f"epoch {epoch} step {step}/{len(train_loader)} "
                      f"loss {loss.item() * grad_accum_steps:.4f}")

        avg_loss = running_loss / len(train_loader)
        epoch_time = time.time() - t0

        # dùng evaluate_detailed() thay vì quick_eval
        eval_result = evaluate_detailed(model, dev_loader, device, verbose=False)
        dev_f1 = eval_result["macro_sentiment_f1_4cls"]

        history.append({"epoch": epoch, "train_loss": avg_loss,
                         "dev_macro_f1": dev_f1, "epoch_time_sec": epoch_time})
        print(f"\n== Epoch {epoch}/{num_epochs} | loss {avg_loss:.4f} "
              f"| dev macro-F1 {dev_f1:.4f} | {epoch_time:.1f}s ==")

        save_checkpoint(checkpoint_path.replace(".pt", "_last.pt"),
                         model, optimizer, scheduler, epoch, best_f1, history,
                         hyperparams or {})

        if dev_f1 > best_f1:
            best_f1 = dev_f1
            patience_counter = 0
            # save_checkpoint(checkpoint_path, model, optimizer, scheduler,
            #                  epoch, best_f1, history, hyperparams or {})
            save_checkpoint(checkpoint_path, model, optimizer, scheduler,
                 epoch, best_f1, history, hyperparams or {},
                 include_optimizer=False)
            print(f"  -> lưu BEST checkpoint tại {checkpoint_path} (F1={best_f1:.4f})")
        else:
            patience_counter += 1
            print(f"  -> không cải thiện ({patience_counter}/{early_stopping_patience})")
            if patience_counter >= early_stopping_patience:
                print("Early stopping.")
                break

    print(f"\nHoàn tất. Best dev macro-F1: {best_f1:.4f}")
    return history

## Evaluate function

In [11]:

# from sklearn.metrics import f1_score
# @torch.no_grad()
# def evaluate(model, loader, device):
#     """Returns macro-F1 averaged across all 10 aspect heads (4-way each)."""
#     model.eval()
#     all_preds = []
#     all_labels = []
#     for batch in loader:
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"]  # keep on cpu for sklearn

#         logits = model(input_ids, attention_mask)  # (B, 10, 4)
#         preds = logits.argmax(dim=-1).cpu()

#         all_preds.append(preds)
#         all_labels.append(labels)

#     all_preds = torch.cat(all_preds, dim=0).numpy()   # (N, 10)
#     all_labels = torch.cat(all_labels, dim=0).numpy()  # (N, 10)

#     per_aspect_f1 = {}
#     for i, aspect in enumerate(ASPECTS):
#         per_aspect_f1[aspect] = f1_score(
#             all_labels[:, i], all_preds[:, i], average="macro", zero_division=0
#         )
#     macro_f1 = float(np.mean(list(per_aspect_f1.values())))
#     return macro_f1, per_aspect_f1
import numpy as np
import torch
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd

@torch.no_grad()
def evaluate_detailed(model, loader, device, verbose=True, show_confusion=False):
    """
    Đánh giá model ABSA đa khía cạnh với nhiều loại chỉ số:

    1. Aspect detection (nhị phân: aspect có được nhắc tới hay không)
       -> Precision / Recall / F1 cho từng aspect
    2. Sentiment classification (4 lớp: None/Positive/Negative/Neutral)
       -> Macro-F1 cho từng aspect
    3. Sentiment chỉ tính trên Positive/Negative (loại None + Neutral)
       -> None quá áp đảo (~70% nhãn) sẽ làm macro-F1 4-lớp "ảo cao";
          chỉ số này cho biết model phân biệt Pos/Neg thật sự tốt tới đâu
    4. Micro-F1 gộp toàn bộ (sample, aspect) pairs — 2 biến thể:
       toàn bộ cặp, và chỉ cặp thực sự có mention (loại None ở cả 2 phía)
    5. Exact-match ratio: tỉ lệ mẫu mà CẢ 10 aspect đều dự đoán đúng
       (chỉ số khắt khe nhất, phản ánh model có "hiểu đúng" câu hay không)

    Returns: dict chứa toàn bộ số liệu (bao gồm DataFrame per-aspect
    và confusion matrix nếu show_confusion=True) + in bảng tổng hợp.
    """
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        logits = model(input_ids, attention_mask)          # (B, 10, 4)
        preds = logits.argmax(dim=-1).cpu()
        all_preds.append(preds)
        all_labels.append(batch["labels"])

    preds = torch.cat(all_preds, dim=0).numpy()   # (N, 10)
    labels = torch.cat(all_labels, dim=0).numpy() # (N, 10)
    n_samples = preds.shape[0]

    rows = []
    confmats = {}

    for i, aspect in enumerate(ASPECTS):
        y_true = labels[:, i]
        y_pred = preds[:, i]

        # (1) Aspect detection: mentioned (1) vs not (0)
        bin_true = (y_true != 0).astype(int)
        bin_pred = (y_pred != 0).astype(int)
        det_p = precision_score(bin_true, bin_pred, zero_division=0)
        det_r = recall_score(bin_true, bin_pred, zero_division=0)
        det_f1 = f1_score(bin_true, bin_pred, zero_division=0)

        # (2) Sentiment 4-class macro-F1 (bao gồm cả None)
        sent_f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

        # (3) Sentiment chỉ tính trên Positive/Negative
        mask_posneg = np.isin(y_true, [1, 2])
        if mask_posneg.sum() > 0:
            posneg_f1 = f1_score(
                y_true[mask_posneg], y_pred[mask_posneg],
                labels=[1, 2], average="macro", zero_division=0,
            )
        else:
            posneg_f1 = float("nan")

        support = int(bin_true.sum())  # số lần aspect này thực sự được nhắc tới

        rows.append({
            "Aspect": aspect, "Support": support,
            "Detect_P": det_p, "Detect_R": det_r, "Detect_F1": det_f1,
            "Sentiment_F1(4cls)": sent_f1_macro,
            "Sentiment_F1(Pos/Neg)": posneg_f1,
        })

        if show_confusion:
            confmats[aspect] = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])

    df = pd.DataFrame(rows)

    # (4) Micro-F1
    micro_f1_all = f1_score(labels.flatten(), preds.flatten(), average="micro", zero_division=0)
    mask_flat = (labels.flatten() != 0) | (preds.flatten() != 0)
    micro_f1_mentioned = f1_score(
        labels.flatten()[mask_flat], preds.flatten()[mask_flat],
        average="micro", zero_division=0,
    )

    # (5) Exact-match ratio
    exact_match = float((preds == labels).all(axis=1).mean())

    summary = {
        "n_samples": n_samples,
        "macro_detection_f1": df["Detect_F1"].mean(),
        "macro_sentiment_f1_4cls": df["Sentiment_F1(4cls)"].mean(),
        "macro_sentiment_f1_posneg": df["Sentiment_F1(Pos/Neg)"].mean(skipna=True),
        "micro_f1_all_pairs": micro_f1_all,
        "micro_f1_mentioned_only": micro_f1_mentioned,
        "exact_match_ratio": exact_match,
        "per_aspect": df,
        "confusion_matrices": confmats if show_confusion else None,
    }

    if verbose:
        pd.set_option("display.float_format", lambda x: f"{x:.4f}")
        print("=" * 78)
        print(f"Đánh giá trên {n_samples} mẫu")
        print("=" * 78)
        print(df.to_string(index=False))
        print("-" * 78)
        print(f"Macro Detection F1               : {summary['macro_detection_f1']:.4f}")
        print(f"Macro Sentiment F1 (4cls)         : {summary['macro_sentiment_f1_4cls']:.4f}")
        print(f"Macro Sentiment F1 (Pos/Neg only) : {summary['macro_sentiment_f1_posneg']:.4f}")
        print(f"Micro F1 (toàn bộ cặp)            : {summary['micro_f1_all_pairs']:.4f}")
        print(f"Micro F1 (chỉ cặp có mention)     : {summary['micro_f1_mentioned_only']:.4f}")
        print(f"Exact-match ratio (10/10 đúng)    : {summary['exact_match_ratio']:.4f}")
        print("=" * 78)

    return summary

## Predict function

In [12]:
@torch.no_grad()
def predict_one(text, model, tokenizer, device):
    processed = preprocess_for_model(clean_text(text))
    encoding = tokenizer(
        processed, truncation=True, max_length=MAX_LEN,
        padding="max_length", return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    logits = model(input_ids, attention_mask)  # (1, 10, 4)
    pred_ids = logits.argmax(dim=-1).squeeze(0).cpu().tolist()

    results = {}
    for aspect, pred_id in zip(ASPECTS, pred_ids):
        polarity = ID2POLARITY[pred_id]
        if polarity != "None":
            results[aspect] = polarity
    return results

# BEGIN WORKFLOW

## Set Seed

In [13]:
def set_seed(seed):
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(True)
set_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)
import os
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # cần cho use_deterministic_algorithms trên CUDA

## Set Device

In [14]:
device = DEVICE if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Set Tokenizer

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

## Set Class Weights

In [16]:
class_weights = compute_class_weights(TRAIN_CSV)
class_weights = {a: w.to(device) for a, w in class_weights.items()}

## Create DataSet and DataLoader

In [17]:
train_ds = ABSADataset(TRAIN_CSV, tokenizer)
test_ds = ABSADataset(TEST_CSV, tokenizer)
dev_ds = ABSADataset(DEV_CSV, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=2,
    pin_memory=True, shuffle=True, generator=g)

dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True, shuffle=True)

test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE)

Word-segmenting (pyvi):   0%|          | 0/7786 [00:00<?, ?it/s]

Word-segmenting (pyvi):   0%|          | 0/2224 [00:00<?, ?it/s]

Word-segmenting (pyvi):   0%|          | 0/1112 [00:00<?, ?it/s]

## Create a model

In [18]:

model = PhoBertABSA().to(device)




import requests
if(CONTINUE_TRAINING):
  url = "https://huggingface.co/NVKQ2022/VN_ABSA_PhoBERT/resolve/main/phobert_absa_state_dict%20%283%29.pth"

  response = requests.get(url)

  print("Status:", response.status_code)
  print("Size:", len(response.content) / 1024**2, "MB")

  with open("/content/test_model.pth", "wb") as f:
      f.write(response.content)

  checkpoint = torch.load(
      "/content/test_model.pth",
      map_location="cpu"
  )

  print("✅ Model loaded successfully")
  print(type(checkpoint))
  print("Number of keys:", len(checkpoint))

  best_model = PhoBertABSA()
  best_model.load_state_dict(checkpoint)
  best_model.to(device)

# result = evaluate_detailed(best_model, test_loader, device, show_confusion=True)



pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

Status: 200
Size: 515.1833124160767 MB
✅ Model loaded successfully
<class 'collections.OrderedDict'>
Number of keys: 219


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Optimizer and LR_Scheduler

In [19]:
optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * NUM_EPOCHS * len(train_loader)),
        num_training_steps=NUM_EPOCHS * len(train_loader),
    )

scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=int(WARMUP_RATIO * NUM_EPOCHS * len(train_loader)),
    num_training_steps=NUM_EPOCHS * len(train_loader),
)
# scheduler = StepLR(optimizer, step_size=2, gamma=0.5)

## Training

In [20]:
import os
os.makedirs("/content/drive/MyDrive/absa_checkpoints", exist_ok=True)
os.path.exists("/content/drive/MyDrive")  # phải trả về True

hyperparams = {"lr": LEARNING_RATE, "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS}

if(CONTINUE_TRAINING) :
  # Re-initialize optimizer and scheduler for best_model
  optimizer_for_best_model = torch.optim.AdamW(
      best_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
  )
  scheduler_for_best_model = get_linear_schedule_with_warmup(
      optimizer_for_best_model,
      num_warmup_steps=int(WARMUP_RATIO * NUM_EPOCHS * len(train_loader)),
      num_training_steps=NUM_EPOCHS * len(train_loader),
  )
  scheduler_for_best_model = get_cosine_schedule_with_warmup(
      optimizer=optimizer_for_best_model,
      num_warmup_steps=int(WARMUP_RATIO * NUM_EPOCHS * len(train_loader)),
      num_training_steps=NUM_EPOCHS * len(train_loader),
  )

  history = train_model(
  best_model,
  train_loader, dev_loader, optimizer_for_best_model, scheduler_for_best_model, device,
  class_weights, num_epochs=NUM_EPOCHS,
  checkpoint_path="/content/drive/MyDrive/absa_checkpoints/best_model.pt",  # lưu thẳng vào Drive
  resume_from="/content/drive/MyDrive/absa_checkpoints/best_model_last.pt",  # None nếu train mới
  hyperparams=hyperparams,
  )

else:
  history = train_model(
      model,
      train_loader, dev_loader, optimizer, scheduler, device,
      class_weights, num_epochs=NUM_EPOCHS,
      checkpoint_path="/content/drive/MyDrive/absa_checkpoints/best_model.pt",  # lưu thẳng vào Drive
      resume_from="/content/drive/MyDrive/absa_checkpoints/best_model_last.pt",  # None nếu train mới
      hyperparams=hyperparams,
  )

epoch 1 step 0/487 loss 0.0646
epoch 1 step 50/487 loss 0.0600
epoch 1 step 100/487 loss 0.0477
epoch 1 step 150/487 loss 0.0751
epoch 1 step 200/487 loss 0.0919
epoch 1 step 250/487 loss 0.0581
epoch 1 step 300/487 loss 0.0875
epoch 1 step 350/487 loss 0.0493
epoch 1 step 400/487 loss 0.0894
epoch 1 step 450/487 loss 0.0550

== Epoch 1/5 | loss 0.0687 | dev macro-F1 0.7394 | 120.6s ==
  -> lưu BEST checkpoint tại /content/drive/MyDrive/absa_checkpoints/best_model.pt (F1=0.7394)
epoch 2 step 0/487 loss 0.0401
epoch 2 step 50/487 loss 0.1055
epoch 2 step 100/487 loss 0.1024
epoch 2 step 150/487 loss 0.0878
epoch 2 step 200/487 loss 0.0592
epoch 2 step 250/487 loss 0.0648
epoch 2 step 300/487 loss 0.0833
epoch 2 step 350/487 loss 0.0616
epoch 2 step 400/487 loss 0.1454
epoch 2 step 450/487 loss 0.0348

== Epoch 2/5 | loss 0.0651 | dev macro-F1 0.7327 | 117.2s ==
  -> không cải thiện (1/3)
epoch 3 step 0/487 loss 0.0638
epoch 3 step 50/487 loss 0.0641
epoch 3 step 100/487 loss 0.0398
epoc

## Evaluating

In [21]:
if(CONTINUE_TRAINING):
  result = evaluate_detailed(best_model, test_loader, device, show_confusion=True)
else :
  result = evaluate_detailed(model, test_loader, device, show_confusion=True)

# result["per_aspect"] là DataFrame, có thể result["per_aspect"].to_csv(...) để lưu lại

Đánh giá trên 2224 mẫu
     Aspect  Support  Detect_P  Detect_R  Detect_F1  Sentiment_F1(4cls)  Sentiment_F1(Pos/Neg)
    BATTERY     1014    0.9525    0.9882     0.9700              0.8434                 0.9428
     CAMERA      588    0.9429    0.9830     0.9625              0.8726                 0.9507
     DESIGN      398    0.8796    0.8995     0.8894              0.7486                 0.8923
   FEATURES      711    0.8586    0.9311     0.8934              0.7743                 0.9125
    GENERAL     1381    0.8852    0.9269     0.9056              0.7828                 0.9266
PERFORMANCE     1172    0.9083    0.9300     0.9191              0.7960                 0.9311
      PRICE      569    0.8900    0.9666     0.9267              0.8384                 0.9098
     SCREEN      269    0.8265    0.9033     0.8632              0.7733                 0.9389
    SER&ACC      593    0.8434    0.8718     0.8574              0.6804                 0.8850
    STORAGE       27    0.7

In [22]:
results = []
print("Predictions for the first 100 samples from the test dataset:")
for i in range(1):
    sample_text = test_ds.texts[i]
    prediction = predict_one(sample_text, model, tokenizer, device)
    results.append(prediction) # Changed from results += prediction
    print(f"Sample {i+1}: {sample_text}\nPrediction: {prediction}\n")

from collections import defaultdict, Counter
def aggregate_results(results):
  """ Tổng hợp nhiều kết quả sentiment. Parameters ---------- results : list[dict] Danh sách các dictionary kết quả. Returns ------- dict Kết quả tổng hợp theo từng category. """
  summary = defaultdict(Counter)
  for result in results:
    for category, sentiment in result.items():
      summary[category][sentiment] += 1
  return dict(summary)

aggregate_results(results)

Predictions for the first 100 samples from the test dataset:


AttributeError: 'ABSADataset' object has no attribute 'texts'

In [ ]:
print(f"{predict_one("May mới mua được 1 tháng. Pin ổn. Chụp hình xấu. Cái viền máy mới đây bị tróc sơn rồi. Quá thất vọng", model, tokenizer, "cuda")}\n ")
print(f"{predict_one("Hàng Sài tạm thì được  không mượt mà lắm . Vi xử lý kém. Nếu chơi game lien quan thì yếu", model, tokenizer, "cuda")} \n ")
print(f"{predict_one("Mình mua tháng từ 12/2017 đến nay dùng vẫn OK, máy chưa vấn đề j hết, pin còn 87%. Rất trâu bò!", model, tokenizer, "cuda")} \n ")



## Checkpoint (not done yet:Đ)

In [ ]:
checkpoint = {
    "training_hyperparameters": {
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
    },
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
}

### Saving Model weight

In [ ]:
# Đường dẫn file model
model_path = 'phobert_absa_state_dict_best.pth'

# Lưu state_dict của model
torch.save(model.state_dict(), model_path)
print(f'Đã lưu model tại: {model_path}')

# Tải file về máy
files.download(model_path)